In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import duckdb as dd
import os

In [ ]:
db_path = "DB/"

customers = pd.read_csv(os.path.join(db_path,"olist_customers_dataset.csv"))
geolocations = pd.read_csv(os.path.join(db_path,"olist_geolocation_dataset.csv"))
order_items = pd.read_csv(os.path.join(db_path,"olist_order_items_dataset.csv"))
order_payments = pd.read_csv(os.path.join(db_path,"olist_order_payments_dataset.csv"))
order_reviews = pd.read_csv(os.path.join(db_path,"olist_order_reviews_dataset.csv"))
orders = pd.read_csv(os.path.join(db_path,"olist_orders_dataset.csv"))
products = pd.read_csv(os.path.join(db_path,"olist_products_dataset.csv"))
sellers = pd.read_csv(os.path.join(db_path,"olist_sellers_dataset.csv"))
product_category_name_translation = pd.read_csv(os.path.join(db_path,"product_category_name_translation.csv"))

In [ ]:
orders_by_month = dd.query(
    """
    SELECT
        EXTRACT(YEAR FROM CAST(order_purchase_timestamp AS TIMESTAMP)) AS year,
        EXTRACT(MONTH FROM CAST(order_purchase_timestamp AS TIMESTAMP)) AS month,
        COUNT(*) AS delivered_by_month
    FROM orders
    WHERE order_status = 'delivered'
    GROUP BY year, month
    ORDER BY year, month
    """
)

orders_by_month_df = orders_by_month.to_df()
orders_by_month_df['month_year'] = orders_by_month_df['month'].astype(str) + '-' + orders_by_month_df['year'].astype(str)

plt.plot(orders_by_month_df['month_year'], orders_by_month_df['delivered_by_month'], marker='o', linestyle='--', color='b')
plt.xlabel('DATE')
plt.ylabel('Delivered Orders')
plt.title('Orders Delivered by Month')
plt.xticks(rotation=60)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
payments_behaviour = dd.query(
    """
    SELECT SUM(payment_value) AS total_payment_value, 
    AVG(payment_value) AS average_payment_value, 
    STDDEV(payment_value) AS standard_deviation, payment_type
    FROM order_payments
    GROUP BY payment_type
    ORDER BY total_payment_value DESC
    """
)

payments_behaviour_df = payments_behaviour.to_df()
#print(payments_behaviour_df)
plt.bar(payments_behaviour_df['payment_type'], payments_behaviour_df['average_payment_value'],
         yerr=payments_behaviour_df['standard_deviation'], color='skyblue')
plt.xlabel('Payment Type')
plt.ylabel('Average Payment Value')
plt.title('Average Payment Value by Payment Type')
plt.xticks(rotation=60)
plt.grid(axis='y')
plt.show()

plt.bar(payments_behaviour_df['payment_type'], payments_behaviour_df['total_payment_value']/payments_behaviour_df['total_payment_value'].sum(), color='lightgreen')
plt.xlabel('Payment Type')
plt.ylabel('Proportion of Total Payment Value')
plt.title('Total Payment Value by Payment Type')
plt.xticks(rotation=60)
plt.grid(axis='y')
plt.show()

In [ ]:
payments_by_order = dd.query(
    """
    SELECT AVG(payment_value) AS avg_payment_by_order, payment_installments
    FROM order_payments
    GROUP BY payment_installments
    ORDER BY avg_payment_by_order DESC
    """
)

payments_by_order_df = payments_by_order.to_df()
#print(payments_by_order_df)
plt.bar(payments_by_order_df['payment_installments'], payments_by_order_df['avg_payment_by_order'], color='lightgreen')
plt.xlabel('Payment Installments')
plt.ylabel('Average Payment Value by Order (R$)')
plt.title('Average Payment Value by Order Installments')
plt.grid(axis='y')
plt.tight_layout()
plt.show()




In [ ]:
most_rentable_states = dd.query(
    """
    SELECT SUM(payment_value) AS total_gains, customer_state
    FROM customers
    INNER JOIN orders
    ON customers.customer_id = orders.customer_id
    INNER JOIN order_payments 
    ON order_payments.order_id = orders.order_id
    WHERE orders.order_status = 'delivered'
    GROUP BY customer_state
    ORDER BY total_gains DESC
    LIMIT 5
    """)

most_rentable_states_df = most_rentable_states.to_df()
plt.bar(most_rentable_states_df['customer_state'], most_rentable_states_df['total_gains'], color='orange')
plt.xlabel('Customer State')
plt.ylabel('Total Gains (R$)')
plt.title('Top 5 Most Rentable States')
plt.grid(axis='y')
plt.show()

In [ ]:
avg_delivery_times = dd.query("""
WITH delivery_times AS (
    SELECT 
        customer_state, 
        CAST(order_purchase_timestamp AS TIMESTAMP) AS purchase_timestamp, 
        CAST(order_delivered_customer_date AS TIMESTAMP) AS delivered_customer_date
    FROM orders
    INNER JOIN customers
        ON orders.customer_id = customers.customer_id
    WHERE order_status = 'delivered')
    
SELECT customer_state, AVG(EPOCH(delivered_customer_date) - EPOCH(purchase_timestamp))/86400 AS average_days_delivery_time, 
STDDEV(EPOCH(delivered_customer_date) - EPOCH(purchase_timestamp))/86400 AS stddev_days_delivery_time
FROM delivery_times
GROUP BY customer_state
ORDER BY average_days_delivery_time DESC
""")
avg_delivery_times = avg_delivery_times.to_df()

print(avg_delivery_times)

plt.bar(avg_delivery_times['customer_state'], avg_delivery_times['average_days_delivery_time'], 
        yerr=avg_delivery_times['stddev_days_delivery_time'], color='purple')
plt.xlabel('Customer State')
plt.ylabel('Average Delivery Time (days)')
plt.title('Average Delivery Time by Customer State')
plt.xticks(rotation=60)
plt.grid(axis='y')
plt.tight_layout()
plt.show()



In [ ]:
delivery_reviews = dd.query("""
    WITH delivery_satisfaction AS (
        SELECT 
            review_score, 
            EPOCH(CAST(order_estimated_delivery_date AS TIMESTAMP)) AS estimated_delivery_timestamp, 
            EPOCH(CAST(order_delivered_customer_date AS TIMESTAMP)) AS delivered_customer_timestamp
        FROM orders
    INNER JOIN order_reviews
        ON orders.order_id = order_reviews.order_id
    WHERE order_status = 'delivered')
    SELECT 
        AVG(review_score) AS average_review_score,
        STDDEV(review_score) AS stddev_review_score,
        CASE WHEN 
                delivered_customer_timestamp - estimated_delivery_timestamp <= 0 THEN 'Early'
                ELSE 'Late' END AS delivery_status
    FROM delivery_satisfaction
    GROUP BY delivery_status
""")
delivery_reviews_df = delivery_reviews.to_df()
print(delivery_reviews_df)

plt.bar(delivery_reviews_df['delivery_status'], delivery_reviews_df['average_review_score'],
        yerr=delivery_reviews_df['stddev_review_score'], color=['green', 'red'])
plt.xlabel('Delivery Status')
plt.ylabel('Average Review Score')
plt.title('Average Review Score by Delivery Status')
plt.xticks(rotation=60)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
most_profitable_categories = dd.query("""
    SELECT 
        product_category_name_translation.product_category_name_english AS product_category_name,
        COUNT(*) AS units_sold_by_category,
        COUNT(DISTINCT orders.order_id) AS total_orders_by_category,
        SUM(order_items.Price) AS total_revenue_by_category
    FROM orders
    INNER JOIN order_items
        ON orders.order_id = order_items.order_id 
    INNER JOIN products
        ON order_items.product_id = products.product_id
    INNER JOIN product_category_name_translation
        ON products.product_category_name = product_category_name_translation.product_category_name
    WHERE orders.order_status = 'delivered' AND products.product_category_name IS NOT NULL
    GROUP BY product_category_name_translation.product_category_name_english
    ORDER BY total_revenue_by_category DESC
    LIMIT 10
    """)


most_profitable_categories_df = most_profitable_categories.to_df()
print(most_profitable_categories_df)

plt.bar(most_profitable_categories_df['product_category_name'], most_profitable_categories_df['total_revenue_by_category'], color='teal')
plt.xlabel('Product Category')
plt.ylabel('Total Revenue by Category (R$)')
plt.title('Total Revenue by Product Category')
plt.xticks(rotation=75)
plt.grid(axis='y')
plt.tight_layout()
plt.show()



In [ ]:
freight_percentage_by_category = dd.query("""
    SELECT 
    product_category_name_translation.product_category_name_english AS product_category_name,
    (SUM(order_items.freight_value) / SUM(order_items.Price + order_items.freight_value))*100 AS freight_percentage 
    FROM products
    INNER JOIN order_items
        ON products.product_id = order_items.product_id
    INNER JOIN orders
        ON order_items.order_id = orders.order_id
    INNER JOIN product_category_name_translation
        ON products.product_category_name = product_category_name_translation.product_category_name
    WHERE orders.order_status = 'delivered' AND product_category_name_translation.product_category_name_english IS NOT NULL
    GROUP BY product_category_name_translation.product_category_name_english
    HAVING COUNT(DISTINCT orders.order_id) > 100
    ORDER BY freight_percentage DESC
    
""")
freight_percentage_by_category_df = freight_percentage_by_category.to_df()

print(freight_percentage_by_category_df)
plt.figure(figsize=(12, 6))
plt.bar(freight_percentage_by_category_df['product_category_name'], freight_percentage_by_category_df['freight_percentage'])
plt.xlabel('Product Category')
plt.ylabel('Freight Percentage')
plt.title('Freight Percentage by Product Category')
plt.xticks(rotation=90, ha='center')
plt.tight_layout()
plt.grid(axis='y')
plt.show()


In [ ]:
best_sellers = dd.query("""
    SELECT 
        SUM(order_items.Price) AS total_revenue, 
        sellers.seller_id,
        AVG(EPOCH(CAST(orders.order_delivered_carrier_date AS TIMESTAMP)) - EPOCH(CAST(orders.order_approved_at AS TIMESTAMP))) / 86400 AS time_to_approve_delivery
    FROM orders
    INNER JOIN order_items
        ON orders.order_id = order_items.order_id
    INNER JOIN sellers
        ON order_items.seller_id = sellers.seller_id
    WHERE orders.order_status = 'delivered'
    GROUP BY sellers.seller_id
    HAVING time_to_approve_delivery < 3
    ORDER BY total_revenue DESC
    LIMIT 3
""").to_df()
print(best_sellers)
